In [0]:
from delta.tables import DeltaTable
import os
import sys

# Ensure Python can find utils.py from the parent directory
project_dir = "/Workspace/Users/ranayuvrajsingh1111@gmail.com/Olist ecomerce ETL"
if project_dir not in sys.path:
  sys.path.append(project_dir)

import utils
from pyspark.sql import functions as F

# ==========================================
# 1. EXTRACT GOLD FACT & SILVER REVIEWS
# ==========================================
print("Extracting data for Seller Performance analytics...")
df_fact = spark.read.table("ecommerce_lakehouse.gold.gold_fact_sales")
df_reviews = spark.read.table("ecommerce_lakehouse.silver.silver_order_reviews")

# Join fact table with order reviews on order_id to bring in review scores
df_fact_with_review = df_fact.join(
    df_reviews.select("order_id", "review_score").dropDuplicates(["order_id"]),
    on="order_id",
    how="left",
)

# ==========================================
# 2. AGGREGATE METRICS BY SELLER
# ==========================================
df_seller_summary = df_fact_with_review.groupBy("seller_key", "seller_state").agg(
    F.countDistinct("order_id").alias("total_orders_handled"),
    F.count("order_item_id").alias("total_items_sold"),
    F.sum("price").alias("total_revenue"),
    F.sum("freight_value").alias("total_freight_collected"),
    F.avg("fulfillment_days").alias("avg_fulfillment_days"),
    F.sum(F.when(F.col("is_delayed") == True, 1).otherwise(0)).alias(
        "delayed_orders_count"
    ),
    F.avg("review_score").alias("avg_review_score"),
)

# Calculate derived metrics like delayed order percentage
df_transformed = df_seller_summary.withColumn(
    "delay_percentage",
    F.round(
        (
            F.col("delayed_orders_count")
            / F.col("total_orders_handled")
            * F.lit(100.0)
        ),
        2,
    ),
).orderBy(F.col("total_revenue").desc())

# ==========================================
# 3. WRITE TO GOLD LAYER
# ==========================================
target_table_name = "ecommerce_lakehouse.gold.gold_seller_performance"
print(f"Writing data to Gold table: {target_table_name}")

df_transformed.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(target_table_name)

print("Successfully generated gold_seller_performance!")

Extracting data for Seller Performance analytics...
Writing data to Gold table: ecommerce_lakehouse.gold.gold_seller_performance
Successfully generated gold_seller_performance!
